# Kite — Kaggle model team (GGUF + llama.cpp)

Notebook này chạy các vai trò LLM trong đội của Kite trên **Kaggle GPU T4 x2**. Mỗi lần chỉ có **một model** được nạp, phục vụ qua `llama-server` (API tương thích OpenAI) bên trong session.

**Lý do không dùng transformers + bitsandbytes như bản trước:**
- Checkpoint gốc 24–32B có dung lượng 48–65 GB, vượt mức đĩa thực tế của Kaggle (~60 GB scratch, `/kaggle/working` ~20 GB).
- Devstral Small 2 được phát hành dưới dạng FP8, mà T4 (Turing, sm_75) không có kernel FP8.
- Gemma 4 có lỗi tràn số khi chạy fp16 trên T4.

GGUF Q4_K_M chỉ nặng khoảng 14–20 GB/model, tải nhanh, và chia được qua 2 GPU. `llama-server` cũng chính là kiểu endpoint mà OpenCode dùng được làm provider sau này.

**Kiểm tra trước khi chạy**
1. Notebook để **private**. Bật **GPU T4 x2** và **Internet**.
2. Không đưa code riêng tư, token hay secret vào ô `TASK`.
3. Chạy lần lượt từ trên xuống. Muốn đổi model thì sửa `TASK["model"]` và chạy lại từ ô **Select model**.
4. Xong việc thì chạy ô **Stop** rồi tắt session (Kaggle không tự tắt GPU một cách tin cậy).

| Model (`TASK["model"]`) | Vai | Ghi chú |
|---|---|---|
| `qwen3.6` | Model chính: phân tích, coding, review, vision | Qwen3.6-35B-A3B, MoE ~3B active, chạy nhanh trên T4 |
| `devstral` | Model thứ hai | Devstral Small 2 24B, dense, chậm hơn |

Model nào làm loại task nào là do bộ điều phối trên EC2 quyết định (dựa trên bảng điểm `routing.model_skill`), không phải notebook này.

**Embedding (BGE-M3 / Nomic) không nằm trong notebook này.** Kaggle không chạy được 24/7: mỗi session tối đa 12 giờ, GPU có quota hằng tuần, và Kite tắt worker khi rảnh. Nên chạy embedding trên EC2 bằng CPU.

## 1. Kiểm tra GPU và đĩa

In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

assert shutil.which("nvidia-smi"), "Chưa bật GPU. Vào Settings -> Accelerator -> GPU T4 x2."
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

# Model lưu ở scratch (không bị giới hạn 20 GB của /kaggle/working, mất khi session tắt).
WORK = Path("/kaggle/tmp/kite")
MODELS_DIR = WORK / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK / "hf")          # cache HF cũng nằm trên scratch
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def free_gb(path=WORK):
    return shutil.disk_usage(path).free / 1024**3

print(f"Free on {WORK}: {free_gb():.1f} GiB")
print(f"Free on /kaggle/working: {shutil.disk_usage('/kaggle/working').free / 1024**3:.1f} GiB")

## 2. Build `llama-server` (CUDA, sm_75)

Lần đầu mất khoảng 5–15 phút. Script chỉ build target `llama-server` cho kiến trúc T4. Commit được dùng sẽ được in ra để ghi lại.

Muốn khỏi build lại: sau lần đầu, copy `/kaggle/tmp/kite/llama.cpp/build/bin` thành một Kaggle Dataset private, rồi trỏ `PREBUILT_BIN` vào đó.

In [ ]:
LLAMA_CPP_REF = "master"          # đổi sang một tag/commit cụ thể khi đã chạy ổn để tái lập được
PREBUILT_BIN = None               # ví dụ: "/kaggle/input/kite-llama-bin/llama-bin"

import glob

def find_libcuda():
    """Tìm libcuda.so cho CMake (target CUDA::cuda_driver).

    Image Kaggle có CUDA 13 nhưng CMake không tự thấy libcuda: ưu tiên file stub của toolkit,
    nếu không có thì dùng driver thật (libcuda.so.1) qua một symlink tên libcuda.so.
    """
    for pattern in ("/usr/local/cuda/lib64/stubs/libcuda.so",
                    "/usr/local/cuda/targets/*/lib/stubs/libcuda.so",
                    "/usr/local/cuda*/lib64/stubs/libcuda.so"):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits[0]
    out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
    real = [line.split("=>")[-1].strip() for line in out.splitlines() if "libcuda.so" in line]
    real += sorted(glob.glob("/usr/lib*/**/libcuda.so*", recursive=True))
    real += sorted(glob.glob("/usr/local/nvidia/lib*/libcuda.so*"))
    real = [r for r in real if Path(r).exists()]
    if not real:
        raise FileNotFoundError("Không tìm thấy libcuda.so. Kiểm tra lại đã bật GPU chưa.")
    link_dir = WORK / "cuda-driver"
    link_dir.mkdir(parents=True, exist_ok=True)
    link = link_dir / "libcuda.so"
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(real[0])
    return str(link)

LLAMA_DIR = WORK / "llama.cpp"
if PREBUILT_BIN and (Path(PREBUILT_BIN) / "llama-server").exists():
    SERVER_BIN = Path(PREBUILT_BIN) / "llama-server"
else:
    SERVER_BIN = LLAMA_DIR / "build" / "bin" / "llama-server"
    if not SERVER_BIN.exists():
        nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
        assert Path(nvcc).exists(), "Không tìm thấy nvcc trong image Kaggle."
        os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
        print(subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout.strip().splitlines()[-1])
        libcuda = find_libcuda()
        print("libcuda cho CMake:", libcuda)
        if not LLAMA_DIR.exists():
            subprocess.run(["git", "clone", "--depth", "50", "https://github.com/ggml-org/llama.cpp", str(LLAMA_DIR)], check=True)
        subprocess.run(["git", "-C", str(LLAMA_DIR), "checkout", LLAMA_CPP_REF], check=True)
        shutil.rmtree(LLAMA_DIR / "build", ignore_errors=True)   # bỏ cache CMake của lần cấu hình lỗi trước
        subprocess.run(["cmake", "-S", str(LLAMA_DIR), "-B", str(LLAMA_DIR / "build"),
                        "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
                        f"-DCMAKE_CUDA_COMPILER={nvcc}",
                        f"-DCUDA_cuda_driver_LIBRARY={libcuda}",
                        "-DLLAMA_CURL=OFF", "-DGGML_CCACHE=OFF",
                        "-DLLAMA_BUILD_TESTS=OFF", "-DCMAKE_BUILD_TYPE=Release"], check=True)
        subprocess.run(["cmake", "--build", str(LLAMA_DIR / "build"), "--config", "Release",
                        "-j", str(os.cpu_count() or 4), "--target", "llama-server"], check=True)

LLAMA_COMMIT = subprocess.run(["git", "-C", str(LLAMA_DIR), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "prebuilt"
print("llama-server:", SERVER_BIN, "| llama.cpp commit:", LLAMA_COMMIT)

## 3. Roster và task

Định dạng mặc định là `Q4_K_M`. `ctx` là 32K: riêng lượt đầu, OpenCode đã gửi khoảng 7.500 token system prompt + định nghĩa tool, nên 16K là không đủ.

In [ ]:
ROSTER = {
    "qwen3.6": {
        "repo": "unsloth/Qwen3.6-35B-A3B-GGUF", "quant": "Q4_K_M",
        "mmproj": True, "ctx": 32768, "max_tokens": 4096,
        # Khuyến nghị của dòng Qwen3 cho chế độ thinking; đo lại trên task thật rồi chỉnh.
        "sampling": {"temperature": 0.6, "top_p": 0.95, "top_k": 20},
        "system_in_user": False, "strip_think": True,
    },
    "devstral": {
        "repo": "unsloth/Devstral-Small-2-24B-Instruct-2512-GGUF", "quant": "Q4_K_M",
        "mmproj": False, "ctx": 32768, "max_tokens": 4096,
        "sampling": {"temperature": 0.15}, "system_in_user": False, "strip_think": False,
    },
}

# Router trên AWS sẽ điền phần này khi có transport được hỗ trợ; hiện tại nhập tay.
TASK = {
    "task_id": "manual-001",
    "model": "qwen3.6",
    "system_prompt": "You are a careful coding assistant. Give a concise, correct answer.",
    "prompt": "Write a Python function unique_preserving_order(items) that removes duplicates while preserving order, plus 3 pytest tests.",
    "image_path": None,   # vision: đường dẫn trong /kaggle/input/...
    "max_tokens": None,   # None = mặc định của model
}

SPEC = ROSTER[TASK["model"]]
print("Model:", TASK["model"], "->", SPEC["repo"], SPEC["quant"])

## 4. Select model: kiểm tra trước rồi mới tải

Ô này hỏi Hugging Face danh sách file và dung lượng **trước khi tải**. Nếu repo/quant không tồn tại hoặc không đủ đĩa, ô sẽ dừng với thông báo rõ ràng. Model cũ được **xóa** trước để nhường đĩa.

In [ ]:
from huggingface_hub import HfApi, snapshot_download

api = HfApi()
info = api.model_info(SPEC["repo"], files_metadata=True)
quant = SPEC["quant"].lower()
weights = sorted((s for s in info.siblings
                  if s.rfilename.endswith(".gguf") and quant in s.rfilename.lower()
                  and "mmproj" not in s.rfilename.lower()),
                 key=lambda s: s.rfilename)
if not weights:
    raise FileNotFoundError(f"{SPEC['repo']} không có file {SPEC['quant']}. Có: "
                            + ", ".join(s.rfilename for s in info.siblings if s.rfilename.endswith('.gguf')))
mmproj = []
if SPEC["mmproj"]:
    projs = [s for s in info.siblings if "mmproj" in s.rfilename.lower() and s.rfilename.endswith(".gguf")]
    projs.sort(key=lambda s: (("f16" not in s.rfilename.lower()), s.size or 0))   # ưu tiên F16
    if not projs:
        raise FileNotFoundError(f"{SPEC['repo']} không có file mmproj cho vision.")
    mmproj = [projs[0]]

need_gb = sum((s.size or 0) for s in weights + mmproj) / 1024**3

# Chỉ giữ một model trên đĩa.
target_dir = MODELS_DIR / SPEC["repo"].replace("/", "__")
for old in MODELS_DIR.iterdir():
    if old != target_dir:
        shutil.rmtree(old, ignore_errors=True)
shutil.rmtree(Path(os.environ["HF_HOME"]), ignore_errors=True)

print(f"Cần tải: {need_gb:.1f} GiB | còn trống: {free_gb():.1f} GiB")
if need_gb + 3 > free_gb():
    raise RuntimeError("Không đủ đĩa. Chọn quant nhỏ hơn (ví dụ Q3_K_M / IQ4_XS) hoặc model nhỏ hơn.")

patterns = [s.rfilename for s in weights + mmproj]
t0 = time.monotonic()
snapshot_download(SPEC["repo"], allow_patterns=patterns, local_dir=target_dir)
MODEL_FILE = target_dir / weights[0].rfilename          # shard đầu tiên; llama.cpp tự nạp các shard còn lại
MMPROJ_FILE = (target_dir / mmproj[0].rfilename) if mmproj else None
print(f"Tải xong sau {time.monotonic() - t0:.0f}s:", MODEL_FILE.name, "|", MMPROJ_FILE.name if MMPROJ_FILE else "no mmproj")

## 5. Khởi động `llama-server` (chỉ loopback)

Server bind vào `127.0.0.1`. **Không mở tunnel/public URL** để AWS gọi vào, đúng với gate an toàn của Kite. Model được chia layer qua 2 GPU.

In [ ]:
import requests

PORT = 8080
BASE = f"http://127.0.0.1:{PORT}"

def stop_server():
    proc = globals().get("SERVER_PROC")
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()

stop_server()
cmd = [str(SERVER_BIN), "-m", str(MODEL_FILE), "--host", "127.0.0.1", "--port", str(PORT),
       "-ngl", "999", "--split-mode", "layer", "-c", str(SPEC["ctx"]),
       "--flash-attn", "auto", "--jinja", "--no-webui"]
if MMPROJ_FILE:
    cmd += ["--mmproj", str(MMPROJ_FILE)]
LOG = WORK / "llama-server.log"
SERVER_PROC = subprocess.Popen(cmd, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

t0 = time.monotonic()
while True:
    if SERVER_PROC.poll() is not None:
        print(LOG.read_text()[-4000:])
        raise RuntimeError("llama-server thoát sớm. Xem log ở trên (thường do OOM hoặc flag không hỗ trợ).")
    try:
        if requests.get(f"{BASE}/health", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if time.monotonic() - t0 > 900:
        raise TimeoutError("Model chưa sẵn sàng sau 15 phút.")
    time.sleep(3)
LOAD_SECONDS = round(time.monotonic() - t0, 1)
print(f"Ready in {LOAD_SECONDS}s")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

## 6. Chạy task

Có thể gọi `chat()` nhiều lần trong cùng session (multi-turn), mỗi lần là một request tới server đang chạy.

In [ ]:
import base64, re

def build_messages(task, spec):
    user_content = task["prompt"]
    if spec["system_in_user"] and task.get("system_prompt"):
        user_content = task["system_prompt"].strip() + "\n\n" + user_content
    if task.get("image_path"):
        if not spec["mmproj"]:
            raise ValueError("Model này không có mmproj cho ảnh; dùng qwen3.6.")
        img = Path(task["image_path"])
        if not img.is_file():
            raise FileNotFoundError(img)
        mime = "image/png" if img.suffix.lower() == ".png" else "image/jpeg"
        data = base64.b64encode(img.read_bytes()).decode()
        user_content = [{"type": "image_url", "image_url": {"url": f"data:{mime};base64,{data}"}},
                        {"type": "text", "text": user_content}]
    messages = []
    if task.get("system_prompt") and not spec["system_in_user"]:
        messages.append({"role": "system", "content": task["system_prompt"]})
    messages.append({"role": "user", "content": user_content})
    return messages

def chat(messages, spec, max_tokens=None):
    body = {"messages": messages, "max_tokens": max_tokens or spec["max_tokens"], **spec["sampling"]}
    r = requests.post(f"{BASE}/v1/chat/completions", json=body, timeout=1800)
    r.raise_for_status()
    return r.json()

started = time.monotonic()
resp = chat(build_messages(TASK, SPEC), SPEC, TASK.get("max_tokens"))
elapsed_seconds = round(time.monotonic() - started, 2)

choice = resp["choices"][0]
raw_text = choice["message"].get("content") or ""
reasoning = choice["message"].get("reasoning_content") or ""
# Một số template trả <think> ngay trong content; tách ra để kết quả gọn.
m = re.search(r"<think>(.*?)</think>", raw_text, flags=re.S)
if m:
    reasoning = reasoning or m.group(1).strip()
    raw_text = raw_text[m.end():]
result_text = raw_text.strip() if SPEC["strip_think"] else (choice["message"].get("content") or "").strip()

usage = resp.get("usage", {})
print(result_text)
print("\n---")
print("finish_reason:", choice.get("finish_reason"), "| usage:", usage, "| seconds:", elapsed_seconds)
if choice.get("finish_reason") == "length":
    print("Cảnh báo: bị cắt do max_tokens. Tăng TASK['max_tokens'] rồi chạy lại ô này.")

## 7. Export kết quả

Đây là phong bì batch chạy tay, **không phải** `WorkerResult` có lease của Kite. Nó không sửa repo và không push.

In [ ]:
response = {
    "schema_version": 1,
    "task_id": TASK["task_id"],
    "model": TASK["model"],
    "model_repo": SPEC["repo"],
    "model_file": MODEL_FILE.name,
    "quant": SPEC["quant"],
    "runtime": {"engine": "llama.cpp", "commit": LLAMA_COMMIT, "ctx": SPEC["ctx"],
                "sampling": SPEC["sampling"], "load_seconds": LOAD_SECONDS},
    "status": "truncated" if choice.get("finish_reason") == "length" else "completed",
    "text": result_text,
    "reasoning_chars": len(reasoning),
    "usage": {"input_tokens": usage.get("prompt_tokens", 0), "output_tokens": usage.get("completion_tokens", 0)},
    "elapsed_seconds": elapsed_seconds,
}
OUTPUT_PATH = Path("/kaggle/working/kite_worker_result.json")
OUTPUT_PATH.write_text(json.dumps(response, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", OUTPUT_PATH)

## 8. Stop

Tắt server và xóa model trên scratch. Sau đó **tắt session** trong giao diện Kaggle (Run → Stop session) để không tốn quota GPU.

In [ ]:
stop_server()
shutil.rmtree(MODELS_DIR, ignore_errors=True)
print("Server stopped, model files removed. Now stop the Kaggle session from the UI.")